# 10 — Practical Decorator Patterns
**Advanced Python · 03_Decorators · Notebook 10/10**

This final notebook combines earlier decorator concepts into reusable patterns. Focus on implementation, trade-offs, and testing rather than repeating introductory theory. All examples run locally; the mini project uses a small CSV and pandas.

## 1. Introduction
Decorators can add logging, timing, validation, retry logic, caching, and access checks without copying the same code into every function. Choose a decorator when behavior genuinely applies to multiple functions; use a regular function when that is clearer.

## 2. Review: Reusable Decorator Template
This template preserves metadata, forwards positional and keyword arguments, and returns the original result.

In [ ]:
from functools import wraps

def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # Before execution
        result = func(*args, **kwargs)
        # After execution
        return result
    return wrapper

## 3. Logging Decorators
Unlike `print()`, `logging` supports levels, handlers, and configurable output. Never log passwords, tokens, or other sensitive arguments/results.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("decorator_examples")

def log_execution(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        logger.info("Calling %s", func.__name__)
        result = func(*args, **kwargs)
        logger.info("Completed %s", func.__name__)
        return result
    return wrapper

@log_execution
def add(a, b):
    return a + b

print(add(10, 20))

## 4. Execution Time Decorators
`time.perf_counter()` is appropriate for elapsed time. `finally` records timing even if the wrapped function raises an exception.

In [ ]:
import time

def measure_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            print(f"{func.__name__}: {elapsed:.4f}s")
    return wrapper

@measure_time
def process_data():
    return sum(range(100_000))

print(process_data())

## 5. Function Call Counter
Store the count on the wrapper. Each decorated function receives its own wrapper and count; the class-based alternative stores it on an instance.

In [ ]:
def count_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.call_count += 1
        return func(*args, **kwargs)
    wrapper.call_count = 0
    return wrapper

@count_calls
def greet(name):
    return f"Hello, {name}"

print(greet("Komal"))
print(greet("Python"))
print("Call count:", greet.call_count)

## 6. Input Validation Decorators
Reusable validation fits a decorator when multiple functions share the same input rule. Keep function-specific business rules inside the function when that is easier to understand.

In [ ]:
def require_positive(func):
    @wraps(func)
    def wrapper(value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("A number is required")
        if value <= 0:
            raise ValueError("Value must be positive")
        return func(value)
    return wrapper

@require_positive
def calculate_square(number):
    return number ** 2

print(calculate_square(5))
try:
    calculate_square(-1)
except ValueError as error:
    print(type(error).__name__, error)

## 7. Retry Decorators
Retries suit *temporary* failures such as transient connection errors. Retry only selected exceptions and take care with operations that may create duplicate transactions or side effects.

In [ ]:
def retry(max_attempts=3, delay=0.01, exceptions=(ConnectionError,)):
    if max_attempts < 1 or delay < 0:
        raise ValueError("max_attempts must be >= 1 and delay >= 0")
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == max_attempts:
                        raise
                    print(f"Attempt {attempt} failed; retrying")
                    time.sleep(delay)
        return wrapper
    return decorator

attempts = {"count": 0}
@retry(max_attempts=3, delay=0)
def fetch_data():
    attempts["count"] += 1
    if attempts["count"] < 3:
        raise ConnectionError("Temporary connection issue")
    return {"status": "ok"}

print(fetch_data())

## 8. Caching and Memoization
`lru_cache` remembers results for repeated calls. Hits reuse a cached result; misses execute the function. `maxsize` bounds retained entries. Arguments must be hashable, and caching is appropriate when results are deterministic and remain valid.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(30))
print(fibonacci.cache_info())

### Simple custom memoization
This small example caches a single hashable positional argument. Prefer `lru_cache` for ordinary production use.

In [ ]:
def memoize_one_argument(func):
    cache = {}
    @wraps(func)
    def wrapper(value):
        if value not in cache:
            cache[value] = func(value)
        return cache[value]
    return wrapper

@memoize_one_argument
def cube(number):
    print("Computing cube")
    return number ** 3

print(cube(4))
print(cube(4))

## 9. Access Control Decorators
This is an **application-level illustration**, not a secure authentication system. In real applications, obtain the current user and roles from a trusted authentication/authorization layer, never from a caller-controlled role argument.

In [ ]:
def require_role(allowed_role):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # Demo only: role comes from a trusted context captured by the app.
            role = trusted_context["role"]
            if role != allowed_role:
                raise PermissionError("Not authorized")
            return func(*args, **kwargs)
        return wrapper
    return decorator

trusted_context = {"role": "analyst"}  # illustrative stand-in for trusted auth
@require_role("analyst")
def view_report():
    return "Report available"

print(view_report())
trusted_context["role"] = "guest"
try:
    view_report()
except PermissionError as error:
    print(error)
trusted_context["role"] = "analyst"

## 10. Rate Limiting Decorators
A simple in-memory sliding-window limiter can restrict calls within one Python process. It is **not** shared across servers/processes, does not persist across restarts, and is not production API rate limiting.

In [ ]:
from collections import deque

def rate_limit(max_calls, period_seconds):
    if max_calls < 1 or period_seconds <= 0:
        raise ValueError("max_calls and period_seconds must be positive")
    def decorator(func):
        timestamps = deque()
        @wraps(func)
        def wrapper(*args, **kwargs):
            now = time.monotonic()
            while timestamps and now - timestamps[0] >= period_seconds:
                timestamps.popleft()
            if len(timestamps) >= max_calls:
                raise RuntimeError("Rate limit exceeded")
            timestamps.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@rate_limit(max_calls=2, period_seconds=60)
def request_preview():
    return "Accepted"

print(request_preview(), request_preview())
try:
    request_preview()
except RuntimeError as error:
    print(error)

## 11. Exception Handling Decorators
Catch only errors for which a fallback is meaningful. Silencing all exceptions can hide bugs.

In [ ]:
def fallback_on_zero_division(default=None):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            try:
                return func(*args, **kwargs)
            except ZeroDivisionError:
                return default
        return wrapper
    return decorator

@fallback_on_zero_division(default=0)
def divide(a, b):
    return a / b

print(divide(10, 0))

## 12. Combining Multiple Decorators
Decoration is applied bottom-up, while the outermost wrapper runs first when the function is called. Order can affect logs, exception handling, and returned values.

In [ ]:
@log_execution
@measure_time
def summarized_total(values):
    return sum(values)

print(summarized_total(range(100_000)))

## 13. Class-Based Practical Decorators
This callable object records both call count and cumulative execution time. See `09_Class_Decorators.ipynb` for the introductory `__init__`/`__call__` explanation.

In [ ]:
from functools import update_wrapper

class ExecutionMonitor:
    def __init__(self, func):
        self.func = func
        self.call_count = 0
        self.total_seconds = 0.0
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        start = time.perf_counter()
        self.call_count += 1
        try:
            return self.func(*args, **kwargs)
        finally:
            self.total_seconds += time.perf_counter() - start

@ExecutionMonitor
def average(values):
    return sum(values) / len(values)

print(average([2, 4, 6]))
print(average([10, 20]))
print("Calls:", average.call_count)
print("Total seconds:", round(average.total_seconds, 6))

## 14. Best Practices
- Use `@wraps` (or `update_wrapper` for callable instances).
- Forward arguments and preserve return values unless intentionally changing behavior.
- Catch specific exceptions and test both success and failure paths.
- Document side effects such as retries, caching, logging, and rate limits.
- Avoid logging sensitive data.
- Prefer a plain function when a decorator adds unnecessary complexity.

**Distinction:** some decorators only surround execution; others deliberately change arguments, results, access, or call frequency.

## 15. Common Mistakes
- Forgetting to return the wrapper or original result.
- Forgetting `*args, **kwargs` forwarding.
- Retrying non-idempotent operations blindly.
- Caching functions with changing external state.
- Using caller-provided roles for authorization.
- Treating a process-local limiter as a distributed limiter.
- Swallowing every exception or logging secrets.
- Assuming decorator stacking order does not matter.

## 16. Mini Project: Data Processing Pipeline
**Goal:** load a small local CSV, validate required columns and values, clean the data, compute summary statistics, and produce an execution report. Reuse logging and timing decorators across processing steps. No paid services or network access are needed.

Install pandas locally if necessary: `pip install pandas`.

In [ ]:
from pathlib import Path
import csv

csv_path = Path("decorator_sales_sample.csv")
rows = [
    {"product": "Pen", "quantity": 2, "price": 10.0},
    {"product": "Book", "quantity": 3, "price": 50.0},
    {"product": "Pen", "quantity": 1, "price": 10.0},
    {"product": "Bag", "quantity": 2, "price": 200.0},
]
with csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["product", "quantity", "price"])
    writer.writeheader()
    writer.writerows(rows)
print("Created:", csv_path)

In [ ]:
import pandas as pd

# Reusable validation decorator for dataframes.
def validate_sales(func):
    @wraps(func)
    def wrapper(df, *args, **kwargs):
        required = {"product", "quantity", "price"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"Missing columns: {sorted(missing)}")
        if df[["product", "quantity", "price"]].isna().any().any():
            raise ValueError("Required fields contain missing values")
        if (df["quantity"] <= 0).any() or (df["price"] < 0).any():
            raise ValueError("Quantity must be positive; price cannot be negative")
        return func(df, *args, **kwargs)
    return wrapper

@log_execution
@measure_time
def load_sales(path):
    return pd.read_csv(path)

@log_execution
@measure_time
@validate_sales
def clean_sales(df):
    clean = df.copy()
    clean["product"] = clean["product"].str.strip()
    clean["revenue"] = clean["quantity"] * clean["price"]
    return clean

@log_execution
@measure_time
@validate_sales
def summarize_sales(df):
    return df.groupby("product", as_index=False).agg(
        units=("quantity", "sum"), revenue=("revenue", "sum")
    )

sales = load_sales(csv_path)
cleaned = clean_sales(sales)
summary = summarize_sales(cleaned)
print(summary)
print("Total revenue:", summary["revenue"].sum())

### Execution report and validation test
The logging decorator records each step; the timing decorator reports elapsed time. Validation is reused for cleaning and summary generation.

In [ ]:
try:
    clean_sales(pd.DataFrame({"product": ["Pen"], "quantity": [1]}))
except ValueError as error:
    print("Validation test:", error)

print("Processed rows:", len(cleaned))
print("Products:", len(summary))
print("Total units:", int(summary["units"].sum()))

## 17. Practice Problems
1. Create a configurable decorator that logs a custom message before execution.
2. Write a timing decorator that returns the original result even when timing is enabled.
3. Implement a call counter and expose its current count.
4. Build a retry decorator that catches a specified exception.
5. Combine logging and timing decorators on a data-processing function; explain the execution order.

## 18. Summary
You can now implement logging, timing, counting, validation, retry, caching, access checks, rate limiting, and exception fallback; combine decorators safely; and choose between function-based and class-based implementations.

**03_Decorators complete — 10/10 notebooks.**

**Next section:** `04_OOP` — Object-Oriented Programming.